# TROPICOR: Quickstart Tutorial
### Orographic Bias Correction & Climatological Validation in Complex Tropical Terrain

Welcome to **TROPICOR**, an open-source Python framework designed to bridge the gap between coarse global climate reanalyses (e.g., ERA5, ERA5-Land) and high-elevation observational networks across complex tropical orography, with a focus on the Colombian Andes.

This quickstart tutorial demonstrates the end-to-end analytical workflow:
1. **Catalog Exploration**: Inspecting Colombian meteorological stations across 6 natural regions.
2. **Synthetic Data**: Generating reproducible tropical hydroclimatic series.
3. **Orographic Correction**: Physics-based lapse-rate elevation correction ($\Gamma = 6.5\,^\circ\text{C}/\text{km}$).
4. **Temporal Alignment & Validation**: Evaluating temperature metrics against ground observations.
5. **Rainfall Climatology & Regime Classification**: Identifying bimodal/unimodal annual cycles.
6. **Thermal Range & Double Difference**: Analyzing Diurnal Temperature Range (DTR) and model thermal dampening.
7. **IDEAM Data Ingestion**: Reading official DHIME station exports.

## 1. Environment Setup
The following cell automatically installs TROPICOR when running inside Google Colab, while cleanly using local development packages in testing or CI environments.

In [1]:
# Install TROPICOR in Google Colab (skipped in local/CI environments)
import sys

if "google.colab" in sys.modules:
    !pip install -q "tropicor==0.1.3"

## 2. Colombian Benchmark Station Catalog
TROPICOR includes a curated benchmark catalog of representative Colombian weather stations spanning diverse elevations and climatic regimes.

In [2]:
from tropicor import NaturalRegion, StationCatalog

catalog = StationCatalog.from_benchmark()
print(f"Loaded {len(catalog)} Colombian benchmark stations across 6 natural regions.")

andina_stations = catalog.filter_by_region(NaturalRegion.ANDINA)
print(f"Andina region stations count: {len(andina_stations)}")
for st in andina_stations[:3]:
    print(f"  - {st.name} ({st.code}): {st.elevation} masl, Dept: {st.department}")

Loaded 48 Colombian benchmark stations across 6 natural regions.
Andina region stations count: 11
  - BOTANA - AUT (52055210): 2790.0 masl, Dept: Nariño
  - JULIA LA (21135050): 1737.0 masl, Dept: Huila
  - JARDIN BOTANICO - AUT (21205710): 2553.0 masl, Dept: Cundinamarca


## 3. Synthetic Tropical Climatology Generation
To ensure this notebook runs deterministically and offline without requiring external API keys or private data files, we generate a 10-year monthly synthetic dataset (2010–2019, 120 steps).

- **Precipitation**: Bimodal annual cycle characteristic of the Colombian inter-Andean valleys, peaking in April and November.
- **Elevation**: Ground station at 2,552 masl (e.g., Bogotá plateau), model grid cell at 1,800 masl (smoothed valley).
- **Thermal Range**: Ground DTR (~12 °C) vs. Reanalysis DTR (~7 °C), illustrating characteristic reanalysis thermal dampening.

In [3]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
dates = pd.date_range("2010-01-01", periods=120, freq="1MS")  # 10 years (2010-2019)

# 1. Andean-like bimodal precipitation series (explicit baseline peaking in Apr & Nov)
# Month:          Jan  Feb  Mar  Apr  May  Jun  Jul  Aug  Sep  Oct  Nov  Dec
monthly_base = np.array([55, 70, 110, 140, 125, 75, 50, 50, 70, 125, 145, 80])
precip_noise = rng.normal(0, 12, size=120)
precip_synthetic = np.clip(monthly_base[dates.month - 1] + precip_noise, 0, None)
precip_series = pd.Series(precip_synthetic, index=dates, name="synthetic_precip")

# 2. Temperature series
month_nums = dates.month.to_numpy()
seasonal_cycle = np.sin(2 * np.pi * (month_nums - 1) / 12)
t_raw_era5 = pd.Series(
    19.0 + 2.0 * seasonal_cycle + rng.normal(0, 0.5, 120),
    index=dates,
)
t_obs_ground = pd.Series(
    14.0 + 1.5 * seasonal_cycle + rng.normal(0, 0.5, 120),
    index=dates,
)

# Thermal extremes: Ground range ~12°C, Model range ~7°C
tmax_obs = t_obs_ground + 6.0 + rng.normal(0, 0.3, 120)
tmin_obs = t_obs_ground - 6.0 + rng.normal(0, 0.3, 120)
tmax_mod = t_raw_era5 + 3.5 + rng.normal(0, 0.3, 120)
tmin_mod = t_raw_era5 - 3.5 + rng.normal(0, 0.3, 120)

print("Synthetic datasets generated successfully:")
print(f"  Dates: {dates[0]:%Y-%m} to {dates[-1]:%Y-%m} ({len(dates)} monthly records)")
print(f"  Mean Precipitation: {precip_series.mean():.1f} mm/month")
print(f"  Raw Reanalysis T:   {t_raw_era5.mean():.1f} °C (Grid elevation: 1800 masl)")
print(f"  Observed Ground T:  {t_obs_ground.mean():.1f} °C (Station: 2552 masl)")

Synthetic datasets generated successfully:
  Dates: 2010-01 to 2019-12 (120 monthly records)
  Mean Precipitation: 90.5 mm/month
  Raw Reanalysis T:   19.0 °C (Grid elevation: 1800 masl)
  Observed Ground T:  14.0 °C (Station: 2552 masl)


## 4. Orographic Lapse-Rate Correction
Coarse atmospheric models smooth out steep topography. If a station sits at 2,552 masl and the model grid is at 1,800 masl, the raw model temperature has a significant warm bias due to hypsometric discrepancies.

TROPICOR applies the physical environmental lapse rate:
$$T_{\text{corrected}} = T_{\text{model}} - \Gamma \cdot (z_{\text{station}} - z_{\text{model}})$$

> **Scientific Note**: In this synthetic example, temperatures are constructed around the standard lapse rate ($\Gamma = 0.0065\,^\circ\text{C}/\text{m}$), yielding near-zero bias by construction. Real-world Andean stations exhibit microclimatic effects, radiative inversions, and residual biases that require local lapse rate calibration or machine learning downscaling.

In [4]:
from tropicor import lapse_rate_temperature_correction

# Always use explicit keyword arguments
t_corrected = lapse_rate_temperature_correction(
    temperature=t_raw_era5,
    station_elevation=2552.0,
    model_elevation=1800.0,
    lapse_rate=0.0065,
)

print(f"Raw Reanalysis Mean T:       {t_raw_era5.mean():.2f} °C")
print(f"Orographic Corrected Mean T: {t_corrected.mean():.2f} °C")
print(f"Observed Ground Mean T:      {t_obs_ground.mean():.2f} °C")

Raw Reanalysis Mean T:       18.98 °C
Orographic Corrected Mean T: 14.10 °C
Observed Ground Mean T:      14.03 °C


## 5. Temporal Alignment & Validation Metrics
TROPICOR provides pairwise temporal alignment (`align_common_period`) and a comprehensive suite of statistical validation metrics.

In [5]:
from tropicor import align_common_period, compute_validation_metrics

# Align to common period (handling missing data pairwise)
aligned_obs, aligned_mod = align_common_period(t_obs_ground, t_corrected)

report = compute_validation_metrics(observed=aligned_obs, modeled=aligned_mod)
print(f"Common timesteps: {report.n_samples}")
print(f"Pearson r:        {report.pearson_r:.3f}")
print(f"RMSE:             {report.rmse:.2f} °C")
print(f"Bias:             {report.bias:.2f} °C")

Common timesteps: 120
Pearson r:        0.834
RMSE:             0.82 °C
Bias:             0.06 °C


## 6. Annual Climatology, Peaks & Regime Classification
The Intertropical Convergence Zone (ITCZ) oscillation produces distinct rainfall regimes in tropical South America. TROPICOR computes monthly climatologies, identifies peaks with prominence filters, and classifies regimes into **unimodal**, **bimodal**, or **complex**.

In [6]:
from tropicor import (
    annual_cycle_peaks,
    classify_rainfall_regime,
    compute_monthly_climatology,
)

clim = compute_monthly_climatology(precip_series)
peaks, proms = annual_cycle_peaks(clim, return_prominences=True)
regime = classify_rainfall_regime(clim)

print("Monthly Climatology:")
for m, val in clim.items():
    bar = "█" * int(val / 8)
    print(f"  Month {m:02d}: {val:5.1f} mm  {bar}")

prom_rounded = [round(p, 1) for p in proms]
print(f"\nDetected Peaks: {peaks} (Prominences: {prom_rounded} mm)")
print(f"Classified Regime: {regime.upper()}")

Monthly Climatology:
  Month 01:  50.5 mm  ██████
  Month 02:  66.3 mm  ████████
  Month 03: 112.3 mm  ██████████████
  Month 04: 142.8 mm  █████████████████
  Month 05: 122.5 mm  ███████████████
  Month 06:  74.6 mm  █████████
  Month 07:  53.2 mm  ██████
  Month 08:  47.8 mm  █████
  Month 09:  69.2 mm  ████████
  Month 10: 122.9 mm  ███████████████
  Month 11: 145.1 mm  ██████████████████
  Month 12:  79.3 mm  █████████

Detected Peaks: [4, 11] (Prominences: [92.3, 97.3] mm)
Classified Regime: BIMODAL


## 7. Diurnal Temperature Range (DTR) & Double Difference

### Scientific Distinction: Mean DTR vs. Monthly Extreme Range
- **Mean DTR** ($\overline{\text{DTR}} = \bar{T}_{\max} - \bar{T}_{\min}$): Daily maximum minus daily minimum, averaged over the month.
- **Monthly Extreme Range** ($\Delta T_{\text{extreme}} = T_{\max, \text{abs}} - T_{\min, \text{abs}}$): Difference between absolute extreme monthly records (e.g. from IDEAM DHIME extremes). Note: $\Delta T_{\text{extreme}} \ge \overline{\text{DTR}}$.

### Model Thermal Dampening
Global reanalyses regularly suffer from **thermal dampening** in high-altitude Andean valleys due to unresolved topography and vegetation representation. The double difference:
$$\Delta R = R_{\text{obs}} - R_{\text{mod}}$$
quantifies this dampening.

In [7]:
from tropicor import (
    compute_double_difference,
    compute_dtr,
    compute_monthly_extreme_range,
)

dtr_obs = compute_dtr(tmax_obs, tmin_obs)
dtr_mod = compute_dtr(tmax_mod, tmin_mod)
double_diff = compute_double_difference(dtr_obs, dtr_mod)

print(f"Mean Observed DTR:            {dtr_obs.mean():.2f} °C")
print(f"Mean Model DTR:               {dtr_mod.mean():.2f} °C")
print(f"Mean Thermal Dampening (ΔDTR): {double_diff.mean():.2f} °C")

# Extreme monthly range calculation:
extreme_range_obs = compute_monthly_extreme_range(tmax_obs, tmin_obs)
print(f"Mean Extreme Range:           {extreme_range_obs.mean():.2f} °C")

Mean Observed DTR:            12.02 °C
Mean Model DTR:               6.96 °C
Mean Thermal Dampening (ΔDTR): 5.06 °C
Mean Extreme Range:           12.02 °C


## 8. Ingesting Real Observational Data (IDEAM DHIME)
To apply TROPICOR to real meteorological stations across Colombia, use `IdeamAdapter` to parse standard Excel/CSV exports from the IDEAM DHIME portal.

In [8]:
from tropicor import IdeamAdapter

# Usage example with local IDEAM DHIME files:
# adapter = IdeamAdapter("path/to/DHIME_export.xlsx")
# obs_series = adapter.get_series()
# print(f"Loaded: {adapter.name} ({adapter.code})")

print(f"IdeamAdapter is available: {IdeamAdapter.__name__}")

IdeamAdapter is available: IdeamAdapter


## 9. Publication Data Visualization & Station Mapping (`tropicor.viz`)

TROPICOR includes a dedicated visualization engine (`tropicor.viz`) engineered for scientific rigor, colorblind accessibility (Okabe & Ito 2008), and offline resilience:
- **Bundled Vector Geometries**: Public-domain Natural Earth boundaries (50m mainland + 10m high-resolution coastlines for San Andrés, Providencia, and Malpelo).
- **Cartographic Maps with Dual Insets**: Dynamically routes outlying archipelago and oceanic island stations into dedicated zoom insets without map clutter.
- **Climatological Comparisons & Regimes**: 12-month annual cycle profiles with integrated peak detection and rainfall regime classification badges.
- **Geometric Taylor Diagrams**: Rigorous synthesis of correlation ($r$), standard deviation ($\sigma$), and centered RMS error ($E'$).
- **Diurnal Thermal Range (DTR)**: Visualizes thermal dampening and double difference bias profiles.

In [9]:
import matplotlib.pyplot as plt

from tropicor import taylor_statistics
from tropicor.viz import (
    plot_climatology_comparison,
    plot_station_map,
    plot_taylor_diagram,
    plot_thermal_range,
)

# 1. Cartographic Station Network Map with Dual Insets
ax_map = plot_station_map(
    catalog,
    color_by_region=True,
    show_insets=True,
    title="TROPICOR Benchmark Meteorological Stations (Colombia)",
    figsize=(7.0, 8.0),
)
plt.show()
print("Rendered national station network with dual insets.")

Rendered national station network with dual insets.


C:\Users\mhgua\AppData\Local\Temp\ipykernel_12580\385668471.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# 2. Climatology Comparison with Automatic Regime Classification
precip_era5 = pd.Series(
    np.maximum(0, precip_series * 0.88 + rng.normal(5, 10, size=len(dates))),
    index=dates,
    name="precip_era5",
)

ax_clim = plot_climatology_comparison(
    observed=precip_series,
    modeled=precip_era5,
    variable="precipitation",
    station_name="Bogota (Andina)",
    model_name="ERA5",
)
plt.show()
print("Generated annual climatology profile with regime badge.")

Generated annual climatology profile with regime badge.


C:\Users\mhgua\AppData\Local\Temp\ipykernel_12580\3495543471.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# 3. Geometric Taylor Diagram Model Verification
t_stats = taylor_statistics(precip_series, precip_era5, normalize=True)
ax_taylor = plot_taylor_diagram(
    t_stats,
    labels=["ERA5 Reanalysis"],
    normalize=True,
    title="Taylor Diagram - Precipitation Cycle Verification",
)
plt.show()
print(f"Computed Taylor stats: r={t_stats.correlation:.2f}, CRMSE={t_stats.crmse:.2f}")

Computed Taylor stats: r=0.95, CRMSE=0.31


C:\Users\mhgua\AppData\Local\Temp\ipykernel_12580\2931101555.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# 4. Thermal Range Diagnostic (DTR)
ax_dtr = plot_thermal_range(
    tmax=tmax_obs,
    tmin=tmin_obs,
    tmax_model=tmax_mod,
    tmin_model=tmin_mod,
    station_name="Bogota (Andina)",
    model_name="ERA5",
)
plt.show()
print("Generated diurnal thermal range and double difference profiles.")

Generated diurnal thermal range and double difference profiles.


C:\Users\mhgua\AppData\Local\Temp\ipykernel_12580\2007692830.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
### Tutorial Complete!
You have successfully explored TROPICOR's complete data pipeline: catalog querying, synthetic data generation, orographic downscaling, statistical validation metrics, annual cycle geometry, diurnal thermal range analysis, and publication-ready cartographic visualization.